# 12 · 3DPW: real outdoor video with 3D ground truth (RQ1 in mm, RQ2/RQ4/RQ5 on real data)

Every model from notebooks 06–08 on the 3DPW **test** split, on identical crops:
released HMR2.0 · B/C/D full fine-tunes (seed 0, RQ2) · experts E0–E3 · generalist GEN (RQ4) ·
DGS weighted / per-joint / learned / top-1 (RQ5). Errors on the 22 SMPL body joints, MPJPE and PA-MPJPE in
mm; comparisons use a bootstrap over **sequences** (frames within a sequence are correlated).

Protocol notes (say these in the report): crops come from the GT joints projected into the image; GT joints
use the neutral SMPL with the GT shape (no gendered models), so absolute numbers are not comparable to
papers (HMR2.0 reports 54.3 PA-MPJPE with the H36M-14 regressor). Comparisons between our models are fair.

3DPW license: non-commercial research only, cite von Marcard et al., ECCV 2018, do not redistribute the raw
data — it stays on the Colab disk; only our results go to Drive.

In [ ]:
STRIDE, BATCH, WORKERS = 2, 48, 8
RUN = "experts_r16_b4_s0"; RQ2_RUN = "ep4_b4_s0"
EXPERTS = ["E0", "E1", "E2", "E3"]

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/CS747_project"
CODE_DIR    = f"{PROJECT_DIR}/code/rw2b"          # the rw2b folder (pose_lib/, blender/, notebooks/)

from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, glob, time, gc, shutil, subprocess
import numpy as np
if not os.path.exists(f"{CODE_DIR}/pose_lib"):              # first run: unpack the code next to the zip
    !cd "{PROJECT_DIR}/code" && unzip -oq rw2b.zip
sys.path.insert(0, CODE_DIR)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# HMR2.0 without PHALP/detectron2 (the install that worked in notebook 05 on Python 3.13)
%cd /content
![ -d 4D-Humans ] || git clone -q https://github.com/shubham-goel/4D-Humans.git
%cd /content/4D-Humans
!pip install -q numpy yacs smplx pytorch_lightning timm einops omegaconf trimesh webdataset pyrender
!pip install -q --no-build-isolation git+https://github.com/mattloper/chumpy
!pip install -q --no-deps .
!sed -i 's/HMR2.load_from_checkpoint(checkpoint_path, strict=False, cfg=model_cfg)/HMR2.load_from_checkpoint(checkpoint_path, strict=False, cfg=model_cfg, weights_only=False)/' hmr2/models/__init__.py
%cd /content
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
sys.path.insert(0, "/content/4D-Humans")

from hmr2.configs import CACHE_DIR_4DHUMANS
from hmr2.models import download_models
download_models(CACHE_DIR_4DHUMANS)                           # ~2.5 GB, cached for the session

# SMPL: plain-numpy files only, chumpy never imported (the fix for the broken environment)
from pose_lib.smpl import pkl_to_npz, setup_hmr2_smpl
SMPL_NPZ = f"{PROJECT_DIR}/data/smpl_neutral.npz"
if not os.path.exists(SMPL_NPZ):
    print(pkl_to_npz(f"{PROJECT_DIR}/body_models/smpl/basicModel_neutral_lbs_10_207_0_v1.0.0.pkl", SMPL_NPZ))
setup_hmr2_smpl(SMPL_NPZ)

import torch
from pose_lib import hmr2_ft as ft, lora, metrics, stats
fk = ft.SMPLJoints(SMPL_NPZ)
print("HMR2.0 + SMPL ready")

## 1. Download 3DPW to the Colab disk (license accepted on the website)

In [ ]:
from pose_lib import threedpw as tp, experts as ex, detector as dt, fusion
ROOT = "/content/3dpw"; os.makedirs(ROOT, exist_ok=True)
for f in tp.FILES:
    if not os.path.exists(f"{ROOT}/{f}"):
        !wget -q -c "{tp.URL}/{f}" -O "{ROOT}/{f}"
    print(f, f"{os.path.getsize(f'{ROOT}/{f}')/1e9:.2f} GB")
if not os.path.exists(f"{ROOT}/sequenceFiles/test"):
    !cd {ROOT} && unzip -oq sequenceFiles.zip
test_seqs = sorted(os.path.basename(f)[:-4] for f in glob.glob(f"{ROOT}/sequenceFiles/test/*.pkl"))
missing = [s for s in test_seqs if not os.path.exists(f"{ROOT}/imageFiles/{s}")]
if missing:
    pats = " ".join(f'"imageFiles/{s}/*"' for s in missing)
    !cd {ROOT} && unzip -oq imageFiles.zip {pats}
print(len(test_seqs), "test sequences")

## 2. Samples (GT joints + crop boxes)

In [ ]:
import cv2
from pose_lib.smpl import SMPLJointsNP
fknp = SMPLJointsNP(SMPL_NPZ)
_wh = {}
def img_wh(seq):
    if seq not in _wh:
        h, w = cv2.imread(f"{ROOT}/imageFiles/{seq}/image_00000.jpg").shape[:2]; _wh[seq] = (w, h)
    return _wh[seq]
S = tp.samples(ROOT, "test", stride=STRIDE, fknp=fknp, img_wh=img_wh)
print(len(S), "samples from", len({s['seq'] for s in S}), "sequences")

## 3. Models

In [ ]:
from hmr2.datasets.vitdet_dataset import ViTDetDataset
from pose_lib import video
model, model_cfg = video.load_hmr2_with_cfg()
MODELS = f"{PROJECT_DIR}/models/{RUN}"
lora.inject(model, lora.hmr2_targets(4), EXPERTS, r=16)
lora.add_adapter(model, "GEN", r=lora.generalist_rank(16, len(EXPERTS)))
for e in EXPERTS + ["GEN"]: lora.load_adapter(model, e, f"{MODELS}/{e}.pt")
model.cuda().eval()
det = dt.Detector("resnet18", pretrained=False).cuda().eval()
det.load_state_dict(torch.load(f"{MODELS}/detector_resnet18.pt", map_location="cuda"))
lf = fusion.LearnedFusion(4); lf.load_state_dict(torch.load(f"{MODELS}/learned_fusion.pt")); lf = lf.cuda().eval()
rq2 = {}
for c in ["B_clean", "C_paste2d", "D_occ3d"]:
    m = ft.load_model(); ft.load_trainable(m, f"{PROJECT_DIR}/models/{RQ2_RUN}/{c}.pt"); rq2[c] = m.cuda().eval()
print("models ready;", torch.cuda.memory_allocated() / 1e9, "GB")

## 4. Inference (resumable: saved to Drive at the end)

In [ ]:
OUT = f"{PROJECT_DIR}/results/3dpw"; os.makedirs(OUT, exist_ok=True)
PRED = f"{OUT}/pred_joints_stride{STRIDE}.npz"

class Crops(torch.utils.data.Dataset):
    def __len__(self): return len(S)
    def __getitem__(self, i):
        s = S[i]; img = cv2.imread(s["img"])
        return torch.as_tensor(ViTDetDataset(model_cfg, img, s["box"][None])[0]["img"]), i

def joints(p):
    return fk(p["global_orient"].float(), p["body_pose"].float(), p["betas"].float()).cpu().numpy()

if not os.path.exists(PRED):
    acc, aux = {}, {k: [] for k in ("cond", "occfrac", "router")}
    dl = torch.utils.data.DataLoader(Crops(), batch_size=BATCH, num_workers=WORKERS)
    fk = fk.cuda(); t0 = time.time()
    for bi, (img, idx) in enumerate(dl):
        img = img.cuda()
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            out = {}
            lora.set_active(model, None); out["hmr2"] = ft.predict(model, img)
            preds = []
            for e in EXPERTS:
                lora.set_active(model, e); p = {k: v.float() for k, v in ft.predict(model, img).items()}
                preds.append(p); out[e] = p
            lora.set_active(model, "GEN"); out["GEN"] = ft.predict(model, img); lora.set_active(model, None)
            d = dt.predict(det, img.float()); w = dt.router_weights(d["cond"])
            out["dgs_weighted"] = fusion.fuse_weighted(preds, w)
            out["dgs_perjoint"] = fusion.fuse_per_joint(preds, w, d["vis"], occ_index=1)
            out["dgs_learned"] = fusion.fuse_joint_weights(preds, lf(d["cond"].float(), d["vis"].float(), d["occfrac"].float()), project="gs")
            out["dgs_top1"] = fusion.fuse_top1(preds, w)
            for c, m in rq2.items(): out[c] = ft.predict(m, img)
        for k, p in out.items(): acc.setdefault(k, []).append(joints(p))
        aux["cond"].append(d["cond"].float().cpu().numpy()); aux["occfrac"].append(d["occfrac"].float().cpu().numpy())
        aux["router"].append(w.float().cpu().numpy())
        if bi % 50 == 0: print(f"{bi * BATCH}/{len(S)}  {time.time() - t0:.0f}s", flush=True)
    np.savez(PRED, **{k: np.concatenate(v).astype(np.float32) for k, v in acc.items()},
             **{"aux_" + k: np.concatenate(v) for k, v in aux.items()},
             gt=np.stack([s["gt"] for s in S]), seq=np.array([s["seq"] for s in S]),
             person=np.array([s["person"] for s in S]), frame=np.array([s["frame"] for s in S]))
    print("saved", PRED, f"{time.time() - t0:.0f}s")

## 5. Results

In [ ]:
import pandas as pd
Z = np.load(PRED); gt, seq = Z["gt"], Z["seq"]
NAMES = ["hmr2", "B_clean", "C_paste2d", "D_occ3d"] + EXPERTS + ["GEN", "dgs_weighted", "dgs_perjoint", "dgs_learned", "dgs_top1"]
E = {}
for n in NAMES:
    e, pa = metrics.joint_errors(Z[n], gt); E[n] = (e.mean(1), pa.mean(1))
tab = pd.DataFrame({n: {"MPJPE": E[n][0].mean(), "PA-MPJPE": E[n][1].mean()} for n in NAMES}).T.round(1)
tab["n"] = len(gt); tab.to_csv(f"{OUT}/summary.csv"); tab

In [ ]:
# Paired comparisons, bootstrap over sequences, Holm-corrected
pairs = [("D_occ3d", "C_paste2d"), ("D_occ3d", "B_clean"), ("C_paste2d", "B_clean"), ("B_clean", "hmr2"),
         ("GEN", "hmr2"), ("dgs_learned", "hmr2"), ("dgs_weighted", "hmr2"), ("dgs_perjoint", "hmr2"),
         ("dgs_learned", "GEN"), ("dgs_weighted", "GEN"), ("E1", "E0"), ("dgs_learned", "dgs_weighted")]
rows = []
for a, b in pairs:
    for k, metric in [(1, "PA-MPJPE"), (0, "MPJPE")]:
        m, lo, hi, p = tp.cluster_bootstrap(E[a][k], E[b][k], seq)
        rows.append({"comparison": f"{a} - {b}", "metric": metric, "diff_mm": round(m, 2), "ci": f"[{lo:+.1f}, {hi:+.1f}]", "p": p})
cmp = pd.DataFrame(rows)
for metric in ["PA-MPJPE", "MPJPE"]:
    mk = cmp.metric == metric; cmp.loc[mk, "p_holm"] = stats.holm(cmp.loc[mk, "p"].values).round(4)
cmp.to_csv(f"{OUT}/comparisons.csv", index=False); cmp

In [ ]:
# Per sequence (PA-MPJPE) and: does the occlusion expert help where the detector sees occlusion? (RQ6 on real)
per_seq = pd.DataFrame({n: pd.Series(E[n][1]).groupby(seq).mean() for n in ["hmr2", "D_occ3d", "GEN", "dgs_learned", "E1"]}).round(1)
per_seq["frames"] = pd.Series(seq).value_counts(); per_seq.to_csv(f"{OUT}/per_sequence_pa.csv"); display(per_seq)
occ = Z["aux_occfrac"]; bins = [0, 0.05, 0.15, 0.3, 1.01]
b = np.digitize(occ, bins) - 1
rows = []
for i in range(len(bins) - 1):
    m = b == i
    if m.sum() < 50: continue
    rows.append({"detector occ fraction": f"{bins[i]:.2f}-{bins[i+1]:.2f}", "frames": int(m.sum()),
                 **{n: round(E[n][1][m].mean(), 1) for n in ["hmr2", "E0", "E1", "GEN", "dgs_learned"]}})
occ_tab = pd.DataFrame(rows); occ_tab.to_csv(f"{OUT}/by_detector_occlusion.csv", index=False); occ_tab